# Lección 6.1 — Por qué combinar modelos funciona: la sabiduría de la multitud

> **Módulo 6 · Ensambles: la sabiduría de la multitud** · Nivel 3 — Practicante
> ⏱️ Tiempo estimado: 80–100 min (lección con ejercicios intercalados ≈ 55 min · ejercicios finales ≈ 30 min)
> 📋 Requisitos previos: Lección 4.5 (sesgo-varianza), Lección 5.4 (árboles de decisión)

### 🎯 Objetivos
Al terminar esta lección podrás:
- Explicar con una analogía por qué el promedio de muchas opiniones suele ganarle a casi cualquier opinión individual.
- Enunciar las **tres condiciones** para que un ensamble funcione: miembros mejores que el azar, errores **distintos** y suficientes miembros.
- Distinguir qué tipo de error arregla un promedio (**varianza**) y cuál no (**sesgo**).
- Ubicar las tres grandes familias de ensambles: **bagging**, **boosting** y **voting/stacking**.

### 🧭 Cómo usar este notebook
| Icono | Significa |
|---|---|
| 📖 | Lee: explicación del concepto |
| ▶️ | Ejecuta la celda y observa el resultado |
| 🔮 | **Predice** antes de ejecutar: escribe tu apuesta y luego compruébala |
| ✋ | **Tu turno**: ejercicio. Las respuestas están ocultas en "👀 Ver respuesta" |
| ✏️ | Modifica un valor en el código y vuelve a ejecutar |

> 💡 No necesitas saber programar. El código está para que **veas** los conceptos en acción; solo tendrás que cambiar números.

## ▶️ Preparación
Ejecuta esta celda una vez (`Shift + Enter`). Carga las herramientas que usaremos.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import binom
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_predict
from sklearn.datasets import make_classification, load_breast_cancer

plt.rcParams["figure.figsize"] = (8, 4.5)
print("✅ Todo listo")

---
## 1. 📖 La intuición: el buey de la feria

En 1906, el científico Francis Galton visitó una feria de ganado en Inglaterra. Había un concurso: **adivinar cuánto pesaría un buey** una vez sacrificado. Participaron unas 800 personas: carniceros, granjeros… y muchísima gente que no sabía nada de ganado.

Galton esperaba demostrar que "la masa" opina mal. Pero al juntar todas las apuestas pasó algo sorprendente:

| | Peso |
|---|---|
| Promedio de las ~800 apuestas | **1,197 libras** |
| Peso real del buey | **1,198 libras** |

El promedio de la multitud se equivocó por **una libra**, y fue mejor que la apuesta de casi cualquier experto individual.

**Un ensamble de modelos hace exactamente esto:** en lugar de confiar en un solo modelo, entrena **muchos** y combina sus predicciones (promediando o votando).

### ✋ Ejercicio 1 — Tres amigos y un frasco de caramelos
Tres amigos adivinan cuántos caramelos hay en un frasco. Hay **530**.

| Amigo | Apuesta |
|---|---|
| Ana | 380 |
| Beto | 690 |
| Carla | 560 |

1. ¿Cuál es el error de cada uno? ¿Y el error **promedio** de los tres?
2. ¿Cuál es el error del **promedio de sus apuestas**?
3. ¿Por qué crees que pasa esto?

✍️ *Tu respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. Errores: Ana 150, Beto 160, Carla 30. Error promedio de una persona: **≈113**.
2. Promedio de apuestas: (380 + 690 + 560) / 3 ≈ 543 → error **≈13**. ¡Mejor que el **mejor** de los tres!
3. Ana se quedó corta y Beto se pasó: **sus errores se cancelan**. Esa cancelación es el secreto de todos los ensambles.
</details>

---
## 2. 📖 El concepto: los errores que se cancelan

Vamos a simular una feria con muchas personas. Cada una mira el buey y apuesta el peso real **más un error al azar**: unas se pasan, otras se quedan cortas.

### 🔮 Ejercicio 2 — Antes de ejecutar
Cada persona se equivoca, en promedio, por unos 60–70 kg. Si promediamos las apuestas de **500 personas**, ¿de qué tamaño crees que será el error del promedio?
- a) Unos 60–70 kg, igual que una persona
- b) Unos 20 kg
- c) Menos de 5 kg

✍️ *Mi apuesta:*

In [ ]:
PESO_REAL = 543          # kg (las 1,198 libras de Galton)
N_PERSONAS = 500         # ✏️ prueba con 5, 50, 5000

rng = np.random.default_rng(0)
apuestas = PESO_REAL + rng.normal(0, 80, N_PERSONAS)    # cada quien se equivoca a su manera

print(f"Error típico de UNA persona : {np.mean(np.abs(apuestas - PESO_REAL)):6.1f} kg")
print(f"Error del PROMEDIO          : {abs(apuestas.mean() - PESO_REAL):6.1f} kg")
print(f"Personas que le ganaron al promedio: {np.mean(np.abs(apuestas - PESO_REAL) < abs(apuestas.mean() - PESO_REAL)):.0%}")

<details><summary><b>👀 Ver respuesta</b></summary>

**c)** Con 500 personas, el error del promedio baja a unos **2 kg**, mientras que una persona típica se equivoca por **≈65 kg**. Solo un **2 %** de las personas le ganó al promedio, y lo hicieron por suerte: no sabrías de antemano quiénes son.

✏️ Con pocas personas el resultado depende mucho de la suerte de quién vino a la feria (con esta semilla: 5 personas → ≈3 kg, 50 personas → ≈10 kg). Con miles, el error del promedio es **siempre** casi nulo. **Más miembros → más cancelación, en promedio.**
</details>

### 🔮 Ejercicio 3 — Cuando todos se equivocan igual
Ahora supón que el buey tiene mucho pelo y **parece** más gordo: a todos les parece unos 40 kg más pesado de lo que es. Ese es un error **compartido** (un *sesgo* común).

Con 5,000 personas, ¿el error del promedio bajará casi a cero como antes?

✍️ *Mi apuesta:*

In [ ]:
SESGO_COMUN = 40         # ✏️ todos sobreestiman esta cantidad (prueba 0, 40, 100)

for n in [5, 50, 500, 5000]:
    apuestas = PESO_REAL + SESGO_COMUN + rng.normal(0, 80, n)
    print(f"{n:>5} personas → error del promedio: {abs(apuestas.mean() - PESO_REAL):5.1f} kg")

<details><summary><b>👀 Ver respuesta</b></summary>

**No.** Por muchas personas que agregues, el error del promedio se queda pegado alrededor de **40 kg** (entre ≈34 y ≈49 en la salida), nunca cerca de cero: el promedio cancela los errores **al azar** (cada uno distinto), pero **no** los errores que **todos comparten**.

Esta es la idea más importante de la lección, en el lenguaje de la Lección 4.5:
- Un promedio reduce la **varianza** (lo que cambia de un miembro a otro).
- Un promedio **no** reduce el **sesgo** (el error que todos tienen en común).
</details>

---
## 3. 📖 Cómo funciona por dentro: el jurado que vota

Con clasificación no promediamos números: **votamos**. Imagina un jurado donde cada miembro acierta el veredicto el **60 %** de las veces, y cada uno decide **por su cuenta** (sin copiarse). Gana la mayoría.

Esta idea tiene más de 200 años: es el **teorema del jurado de Condorcet** (1785). La celda calcula la probabilidad de que la **mayoría** acierte.

### 🔮 Ejercicio 4 — ¿Qué tan bueno es un jurado de "mediocres"?
Cada miembro acierta el 60 %. ¿Qué exactitud esperas de la mayoría con **11** miembros? ¿Y con **101**?

✍️ *Mis apuestas:* 11 → ___ % · 101 → ___ %

In [ ]:
P_ACIERTO = 0.60         # ✏️ prueba 0.51, 0.70 y luego 0.45

for n in [1, 11, 101, 1001]:
    # probabilidad de que MÁS de la mitad acierte (votantes independientes)
    p_mayoria = 1 - binom.cdf(n // 2, n, P_ACIERTO)
    print(f"Jurado de {n:>4} → la mayoría acierta el {p_mayoria:6.1%} de las veces")

<details><summary><b>👀 Ver respuesta</b></summary>

- 11 miembros → **≈75 %**.
- 101 miembros → **≈98 %**.
- 1,001 miembros → **≈100 %**.

Miembros que solo son "un poco mejores que lanzar una moneda", juntos, se vuelven casi infalibles… **si sus errores son independientes**.

✏️ Con `P_ACIERTO = 0.45` pasa lo contrario: el jurado **empeora** al crecer (con 1,001 miembros la mayoría casi nunca acierta). **Juntar modelos peores que el azar amplifica el error.**
</details>

### 📖 La trampa: ¿y si se copian?
El cálculo anterior supone que cada miembro piensa **por su cuenta**. En la vida real, los miembros de un jurado se influyen. Vamos a simularlo: hay un "líder de opinión" y, en cada votación, cada miembro **copia al líder** con cierta probabilidad (`COPIA`). El resto de las veces decide por sí mismo.

### 🔮 Ejercicio 5 — Jurados que se copian
Jurado de 101 miembros, cada uno con 60 % de acierto. Si se copian del líder **la mitad de las veces** (`COPIA = 0.5`), ¿la exactitud de la mayoría estará más cerca del 98 % (independientes) o del 60 % (una sola persona)?

✍️ *Mi apuesta:*

In [ ]:
def exactitud_jurado(n_miembros, p=0.6, copia=0.0, n_votaciones=20_000, semilla=0):
    rng = np.random.default_rng(semilla)
    lider = rng.random(n_votaciones) < p                        # ¿acierta el líder en cada votación?
    propio = rng.random((n_miembros, n_votaciones)) < p          # opinión propia de cada miembro
    copia_al_lider = rng.random((n_miembros, n_votaciones)) < copia
    aciertos = np.where(copia_al_lider, lider, propio)
    return np.mean(aciertos.sum(axis=0) > n_miembros / 2)        # ¿ganó la respuesta correcta?

for copia in [0.0, 0.2, 0.5, 0.8, 1.0]:      # ✏️
    print(f"COPIA = {copia:.1f} → exactitud del jurado de 101: {exactitud_jurado(101, copia=copia):.1%}")

<details><summary><b>👀 Ver respuesta</b></summary>

Mucho más cerca del **60 %**: con `COPIA = 0.5` el jurado de 101 acierta apenas **≈60 %**. Ya con un 20 % de copia cae al **≈73 %**, lejísimos del 98 %. Y con `COPIA = 1.0` el jurado **es** el líder: 101 votos, una sola opinión.

Moraleja: **lo que importa no es cuántos miembros hay, sino cuántas opiniones distintas hay.**
</details>

### 📖 Las tres condiciones de un buen ensamble

| # | Condición | Si no se cumple… | En la feria |
|---|---|---|---|
| 1 | Cada miembro es **mejor que el azar** | El ensamble empeora al crecer | Las apuestas no son disparates |
| 2 | Los miembros se equivocan **de forma distinta** (diversidad) | Votar no cancela nada | Cada persona apuesta sin ver a las demás |
| 3 | Hay **suficientes** miembros | Poca cancelación | 800 personas, no 3 |

Todo el Módulo 6 trata de cómo conseguir la condición 2 a propósito: **¿cómo entreno muchos modelos que se equivoquen de forma distinta?**

### ✋ Ejercicio 6 — ¿Funcionará este ensamble?
Para cada caso, di qué condición falla (si alguna):
1. Diez copias del **mismo** árbol, entrenadas con los **mismos** datos y la misma semilla.
2. 500 modelos con 48 % de exactitud en un problema de dos clases balanceadas.
3. Una regresión logística, un KNN y un árbol, cada uno con ~85 % de exactitud.
4. Dos modelos excelentes y distintos (ensamble de 2 que vota por mayoría).

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. Falla la **2**: diez copias idénticas cometen exactamente los mismos errores. Votar no cambia nada (lo comprobarás en la Sección 6).
2. Falla la **1**: son peores que lanzar una moneda; juntarlos empeora.
3. **Funciona**, en principio: son mejores que el azar y, como "piensan" distinto (rectas, vecinos, preguntas), sus errores no coinciden del todo.
4. Falla la **3** de una forma curiosa: con 2 votantes, cuando no están de acuerdo hay un **empate** y la mayoría no decide nada. Por eso se usa un número impar de votantes… o se promedian probabilidades (Lección 6.5).
</details>

---
## 4. 📖 Qué pasa si… promediamos árboles

Volvamos a los modelos. En la Lección 5.4 viste que un árbol sin límites **memoriza** el ruido: tiene poco sesgo pero **mucha varianza** (si cambias un poco los datos, cambia por completo).

¿Qué pasa si entrenamos **muchos** árboles, cada uno con una versión un poco distinta de los datos, y **promediamos** sus predicciones? Para que cada árbol vea datos distintos, a cada uno le damos una muestra tomada **con reemplazo** de los datos originales (algunos ejemplos se repiten y otros faltan; lo verás en detalle en la Lección 6.2).

Usaremos datos con forma de ola (la curva real es una función seno) más ruido.

In [ ]:
rng_ola = np.random.default_rng(0)
x = np.sort(rng_ola.uniform(0, 10, 80))
y = np.sin(x) + rng_ola.normal(0, 0.4, 80)          # 80 puntos ruidosos
x_fino = np.linspace(0, 10, 500)                     # donde dibujaremos las predicciones

def promedio_de_arboles(n_arboles, max_depth):
    rng = np.random.default_rng(1)
    predicciones = []
    for i in range(n_arboles):
        muestra = rng.integers(0, len(x), len(x))    # muestra con reemplazo
        arbol = DecisionTreeRegressor(max_depth=max_depth, random_state=i)
        arbol.fit(x[muestra, None], y[muestra])
        predicciones.append(arbol.predict(x_fino[:, None]))
    return np.mean(predicciones, axis=0)

def error_vs_ola_real(pred):
    return np.mean((pred - np.sin(x_fino)) ** 2)    # qué tan lejos está de la ola verdadera

### 🔮 Ejercicio 7 — Uno contra cien
Vamos a comparar **1 árbol profundo** (sin límite) contra el **promedio de 100 árboles profundos**.
1. ¿Cuál crees que dibujará una curva más suave?
2. ¿Cuál quedará más cerca de la ola real?

✍️ *Mis predicciones:*

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, n in zip(axes, [1, 100]):
    pred = promedio_de_arboles(n, max_depth=None)
    ax.scatter(x, y, s=12, c="gray", label="datos")
    ax.plot(x_fino, np.sin(x_fino), "k--", label="ola real")
    ax.plot(x_fino, pred, "r", lw=2, label="predicción")
    ax.set_title(f"{n} árbol(es) profundo(s) · error = {error_vs_ola_real(pred):.3f}")
    print(f"{n:>3} árbol(es) profundo(s) → error frente a la ola real: {error_vs_ola_real(pred):.3f}")
axes[0].legend(); plt.tight_layout(); plt.show()

<details><summary><b>👀 Ver respuesta</b></summary>

1. El **promedio de 100** es mucho más suave. Cada árbol profundo, solo, salta de punto en punto persiguiendo el ruido.
2. El promedio también queda **mucho más cerca** de la ola: su error baja de **≈0.17** a **≈0.08**, menos de la mitad.

Cada árbol memoriza **su propia** muestra, con su ruido particular. Al promediar, los "saltos" de unos se cancelan con los de otros: **el promedio reduce la varianza**. Esta es, literalmente, la idea de **bagging** y **Random Forest** (Lección 6.2).
</details>

### 🔮 Ejercicio 8 — ¿Y si promediamos tocones?
Ahora usaremos árboles de **una sola pregunta** (`max_depth=1`, los *tocones* de la Lección 5.4). Un tocón solo puede partir la recta en dos escalones: tiene **mucho sesgo**.

¿Promediar 100 tocones arreglará el problema y dibujará la ola?

✍️ *Mi predicción:*

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, n in zip(axes, [1, 100]):
    pred = promedio_de_arboles(n, max_depth=1)      # ✏️ prueba max_depth=3
    ax.scatter(x, y, s=12, c="gray")
    ax.plot(x_fino, np.sin(x_fino), "k--")
    ax.plot(x_fino, pred, "r", lw=2)
    ax.set_title(f"{n} tocón(es) · error = {error_vs_ola_real(pred):.3f}")
    print(f"{n:>3} tocón(es) → error frente a la ola real: {error_vs_ola_real(pred):.3f}")
plt.tight_layout(); plt.show()

<details><summary><b>👀 Ver respuesta</b></summary>

**No.** El promedio de 100 tocones sigue siendo una curva tosca, muy lejos de la ola (error **≈0.32**, apenas mejor que un tocón solo, **≈0.36**). Todos los tocones cometen **el mismo** error de fondo: son demasiado simples. Es el "buey peludo" de la Sección 2: un **sesgo compartido** que el promedio no puede cancelar.

¿Entonces los modelos simples no sirven para ensambles? Sí sirven, pero hay que combinarlos de otra forma: en lugar de promediar tocones independientes, entrenarlos **uno detrás de otro**, cada uno corrigiendo lo que falló el anterior. Eso es **boosting** (Lección 6.3), y ataca precisamente el **sesgo**.
</details>

### 📖 Ahora con clasificación: un comité de tres modelos distintos
Otra forma de conseguir diversidad es usar modelos que **piensan distinto**: una regresión logística (traza rectas, Lección 5.2), un KNN (mira a los vecinos, Lección 5.3) y un árbol (hace preguntas, Lección 5.4). Cada uno vota y gana la mayoría.

Usaremos 600 ejemplos sintéticos con 6 características (la mitad para entrenar, la mitad para probar).

In [ ]:
X_sint, y_sint = make_classification(n_samples=600, n_features=6, n_informative=4, n_redundant=0,
                                     flip_y=0.05, random_state=4)
X_tr, X_te, y_tr, y_te = train_test_split(X_sint, y_sint, test_size=0.5, random_state=4)

comite = {
    "Logística": LogisticRegression(),
    "KNN (k=15)": KNeighborsClassifier(n_neighbors=15),
    "Árbol (prof. 5)": DecisionTreeClassifier(max_depth=5, random_state=0),
}
votos = pd.DataFrame({nombre: m.fit(X_tr, y_tr).predict(X_te) for nombre, m in comite.items()})
votos["Mayoría"] = (votos.sum(axis=1) >= 2).astype(int)    # 2 de 3 votos ganan

print("Exactitud en prueba:")
print(pd.Series((votos.values == y_te[:, None]).mean(axis=0), index=votos.columns).round(3))

### ✋ Ejercicio 9 — ¿Ganó el comité?
1. ¿La mayoría le gana a **los tres** miembros, a algunos o a ninguno?
2. Mira la tabla de abajo: ¿en cuántos puntos se equivocan los tres a la vez? ¿Puede el voto arreglar esos puntos?

✍️ *Mis respuestas:*

In [ ]:
errores = votos[list(comite)].values != y_te[:, None]       # True donde cada modelo se equivoca
cuantos_fallan = errores.sum(axis=1)
print(pd.Series(cuantos_fallan).value_counts().sort_index().rename("puntos")
      .rename_axis("modelos que fallan"))

<details><summary><b>👀 Ver respuesta</b></summary>

1. La mayoría (**0.890**) queda **por encima de los tres** miembros (0.863–0.873). No es una ganancia gigantesca, pero es gratis: no hubo que inventar ningún modelo nuevo.
2. Hay **15** puntos donde fallan **los tres** a la vez y 18 donde fallan dos: ahí el voto **no puede** hacer nada. En cambio, en los **37** puntos donde falla **uno solo**, los otros dos lo corrigen. Por eso la diversidad importa tanto: cuantos menos errores compartan, más puede rescatar el comité.
</details>

---
## 5. 📖 El mapa del módulo: tres formas de armar la multitud

```
                  ┌─────────────────────────────────────────────────────────┐
                  │                       ENSAMBLES                         │
                  └─────────────────────────────────────────────────────────┘
        BAGGING (6.2)               BOOSTING (6.3–6.4)            VOTING / STACKING (6.5)
   muchos modelos iguales        modelos en fila: cada uno       modelos DISTINTOS que
   en PARALELO, cada uno con     corrige los errores del         votan, o un "jefe" que
   datos un poco distintos       anterior                        aprende a quién creerle
   ┌──┐ ┌──┐ ┌──┐ ┌──┐            ┌──┐→┌──┐→┌──┐→┌──┐              ┌────┐ ┌───┐ ┌─────┐
   │🌳│ │🌳│ │🌳│ │🌳│            │🌱│ │🌱│ │🌱│ │🌱│              │ 📈 │ │👥 │ │ 🌳  │
   └──┘ └──┘ └──┘ └──┘            └──┘ └──┘ └──┘ └──┘              └────┘ └───┘ └─────┘
     ↘    ↓    ↓   ↙                        ↓                          ↘    ↓    ↙
       promedio/voto               suma de todas las correcciones      voto / meta-modelo
   ⬇️ reduce VARIANZA              ⬇️ reduce SESGO                      aprovecha la diversidad
   Ej.: Random Forest              Ej.: AdaBoost, Gradient Boosting,    Ej.: VotingClassifier,
                                   XGBoost, LightGBM, CatBoost          StackingClassifier
```

### ✋ Ejercicio 10 — ¿Qué familia es?
1. "Entreno 300 árboles profundos, cada uno con una muestra distinta de los datos, y promedio."
2. "Entreno un árbol pequeño; luego otro que se concentra en los ejemplos que el primero falló; luego otro…"
3. "Junto una logística, un SVM y un Random Forest, y una regresión logística final aprende cuánto confiar en cada uno."
4. ¿Cuál de las tres familias usarías para arreglar un modelo que **sobreajusta**? ¿Y uno que **subajusta**?

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. **Bagging** (de hecho, casi un Random Forest).
2. **Boosting**.
3. **Stacking** (el modelo final es el "jefe").
4. Sobreajuste = mucha **varianza** → **bagging** (promediar cancela la varianza, como en el Ejercicio 7). Subajuste = mucho **sesgo** → **boosting** (sumar correcciones permite construir un modelo más complejo, algo que el Ejercicio 8 mostró que el promedio no logra).
</details>

---
## 6. 📖 Errores comunes y señales de alerta

| 🚨 Síntoma | Causa probable | Qué probar |
|---|---|---|
| El ensamble rinde **igual** que su mejor miembro | Los miembros se equivocan en los mismos ejemplos (poca diversidad) | Usar datos distintos (6.2), características distintas (6.2) o modelos de familias distintas (6.5) |
| El ensamble rinde **peor** que su mejor miembro | Hay miembros muy malos que votan con el mismo peso | Quitar los malos, dar pesos (6.5) o usar stacking |
| Promediar muchos modelos simples no mejora nada | Su problema es **sesgo**, no varianza | Usar boosting (6.3) o miembros más complejos |
| "Mi ensamble tiene 100 % en entrenamiento" | Evaluaste con datos que los miembros ya vieron | Evaluar siempre con validación cruzada o prueba (Lección 4.6) |
| El modelo es lento y nadie lo entiende | Un ensamble de 1,000 modelos para ganar 0.1 % | Comparar contra un modelo simple; ¿vale la pena la ganancia? |

### 🔮 Ejercicio 11 — El ensamble de clones
Entrenamos **cinco** árboles idénticos (mismos datos, mismos hiperparámetros, misma semilla) y los ponemos a votar. ¿Qué exactitud tendrá el voto comparada con la de un solo árbol?

✍️ *Mi predicción:*

In [ ]:
clones = [DecisionTreeClassifier(max_depth=5, random_state=0).fit(X_tr, y_tr) for _ in range(5)]
votos_clones = np.array([c.predict(X_te) for c in clones])
voto = (votos_clones.sum(axis=0) >= 3).astype(int)

print(f"Un árbol          : {np.mean(clones[0].predict(X_te) == y_te):.3f}")
print(f"Voto de 5 clones  : {np.mean(voto == y_te):.3f}")
print(f"¿Los 5 predicen exactamente lo mismo? {bool((votos_clones == votos_clones[0]).all())}")

<details><summary><b>👀 Ver respuesta</b></summary>

**Exactamente la misma.** Los cinco clones hacen las mismas predicciones, así que el voto es una copia del primer árbol. Cinco veces el costo, cero ganancia. Es la condición 2 rota por completo (`COPIA = 1.0` del Ejercicio 5).
</details>

---
## 7. 📋 Ficha de la idea: ensamble

| | Ensamble de modelos |
|---|---|
| **Qué es** | Combinar las predicciones de varios modelos (promedio, voto o un meta-modelo) |
| **Intuición** | La sabiduría de la multitud: errores distintos se cancelan |
| **Por qué funciona** | Promediar reduce la **varianza**; entrenar en secuencia (boosting) reduce el **sesgo** |
| **Condiciones** | Miembros mejores que el azar · errores distintos (diversidad) · suficientes miembros |
| **✅ Fortalezas** | Casi siempre más preciso y más estable que un solo modelo |
| **❌ Debilidades** | Más lento, más pesado y más difícil de explicar que un solo modelo |
| **Úsalo cuando…** | Necesitas la mayor precisión posible, sobre todo con datos tabulares (Lección 6.6) |
| **Evítalo cuando…** | Necesitas un modelo que se pueda leer entero, o la ganancia es mínima frente al costo |

## 8. 🧠 Resumen en 5 puntos
1. Combinar modelos funciona por la misma razón que el promedio de la feria: los **errores distintos se cancelan**.
2. Hacen falta tres condiciones: miembros **mejores que el azar**, que se equivoquen **de forma distinta** y que sean **suficientes**.
3. La **diversidad** es lo que más importa: 101 miembros que se copian valen lo mismo que uno.
4. Promediar reduce la **varianza** (árboles profundos que sobreajustan), pero **no** el **sesgo** compartido (tocones que subajustan).
5. Hay tres familias: **bagging** (paralelo, contra la varianza), **boosting** (en fila, contra el sesgo) y **voting/stacking** (modelos distintos).

---
## 9. 🏋️ Ejercicios finales

Ya practicaste cada idea por separado. Ahora toca **combinarlas**. Estos ejercicios son más largos (≈30 min).

### 9.1 Repaso rápido (opción múltiple)

**1.** ¿Qué tipo de error reduce **promediar** muchos modelos entrenados con datos ligeramente distintos?
- a) El sesgo · b) La varianza · c) Los dos por igual · d) Ninguno

**2.** Un jurado de 1,001 miembros independientes, cada uno con 52 % de acierto. La mayoría acertará…
- a) ≈52 % · b) ≈50 % · c) ≈90 % · d) ≈100 %

**3.** ¿Cuál de estos ensambles tiene **menos** probabilidades de mejorar sobre sus miembros?
- a) 3 modelos de familias distintas con ~80 % cada uno
- b) 100 árboles profundos con muestras distintas de los datos
- c) 100 copias del mismo modelo con los mismos datos
- d) 50 tocones entrenados uno detrás de otro corrigiendo errores

**4.** Tu modelo tiene 70 % en entrenamiento y 69 % en prueba (subajuste). ¿Qué familia de ensambles atacaría mejor ese problema?
- a) Bagging · b) Boosting · c) Ninguna, los ensambles solo sirven contra el sobreajuste

**5.** En un comité de 3 modelos que votan, ¿en qué puntos **no** puede ayudar el voto?
- a) Donde falla uno solo · b) Donde fallan dos o tres · c) En ninguno · d) En todos

✍️ *Mis respuestas:* 1 → _ · 2 → _ · 3 → _ · 4 → _ · 5 → _

<details><summary><b>👀 Ver respuestas</b></summary>

1. **b)** Los errores al azar de cada modelo se cancelan; un sesgo compartido no.
2. **c)** Con `P_ACIERTO = 0.52` y 1,001 miembros, la celda del Ejercicio 4 da ≈90 %. ¡Una ventaja minúscula multiplicada por mil se vuelve enorme! (si son independientes).
3. **c)** Copias idénticas cometen los mismos errores: cero diversidad.
4. **b)** El subajuste es sesgo; el boosting construye un modelo más complejo sumando correcciones.
5. **b)** Si fallan dos o tres, la mayoría falla. El voto solo rescata los puntos donde falla uno.
</details>

### 9.2 Preguntas para pensar (sin ejecutar)

**A.** En la feria de Galton, ¿por qué crees que habría salido **peor** el promedio si las apuestas se hubieran gritado en voz alta, una tras otra, en lugar de escribirse en papeletas secretas?

**B. Caso de análisis 💼** Un hospital tiene tres modelos para detectar una enfermedad, cada uno con 90 % de exactitud. Los tres se entrenaron con **los mismos datos** y usan **las mismas** cinco variables. El director propone combinarlos por voto "para ganar precisión". ¿Qué le dirías? ¿Qué le pedirías revisar antes de invertir en eso?

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuestas</b></summary>

**A.** Porque las apuestas dejarían de ser **independientes**: los que hablan después se dejan influir por los primeros (el "líder de opinión" del Ejercicio 5). Los errores se vuelven parecidos y ya no se cancelan. Las papeletas secretas protegían la **diversidad**.

**B.** Que el voto solo ayuda si los modelos **se equivocan en pacientes distintos**. Con mismos datos y mismas variables, lo más probable es que fallen en los mismos casos y el comité rinda igual que un solo modelo. Antes de invertir, pediría:
- Una tabla como la del Ejercicio 9: ¿en cuántos pacientes fallan los tres a la vez?
- Medir el comité con validación cruzada, no con los datos de entrenamiento.
- Si hay poca diversidad: probar con modelos de familias distintas o con datos/variables distintas.
</details>

### 9.3 🧩 Práctica integradora: el comité de diagnóstico

Volvemos a los datos de cáncer de mama de la Lección 5.4 (569 pacientes). Formarás un comité de modelos y medirás si de verdad gana, y **por qué**.

Para que los números sean confiables usaremos `cross_val_predict` (Lección 4.6): cada paciente recibe la predicción de un modelo que **no lo vio** al entrenar.

Lo harás en 5 pasos cortos. Cada uno tiene su respuesta: **no avances sin revisarla**.

**Paso 1 ▶️ — Tres miembros distintos**

In [ ]:
X_cm, y_cm = load_breast_cancer(return_X_y=True)

miembros = {
    "Logística": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    "KNN":       make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=15)),
    "Árbol":     DecisionTreeClassifier(max_depth=3, random_state=0),
}
pred = pd.DataFrame({n: cross_val_predict(m, X_cm, y_cm, cv=5) for n, m in miembros.items()})
pd.Series((pred.values == y_cm[:, None]).mean(axis=0), index=pred.columns).round(3)

**Paso 2 🔮 — ¿Le ganará el comité al mejor?** Mira la exactitud de los tres. Antes de ejecutar: ¿el voto por mayoría superará a la **logística**?

✍️ *Mi apuesta:*

In [ ]:
def exactitud_voto(pred, columnas):
    voto = (pred[columnas].sum(axis=1) > len(columnas) / 2).astype(int)
    return np.mean(voto == y_cm)

print(f"Voto Logística + KNN + Árbol: {exactitud_voto(pred, ['Logística', 'KNN', 'Árbol']):.3f}")

<details><summary><b>👀 Ver respuesta</b></summary>

La logística sola tiene ≈0.981; KNN ≈0.961 y el árbol ≈0.917. El voto da ≈**0.965**: le gana a KNN y al árbol, pero **pierde** contra la logística. Cuando un miembro es claramente mejor que los demás, darles el mismo voto a los peores puede **arrastrar** al comité hacia abajo (segunda fila de la tabla de errores comunes). Ganarle al **mejor** miembro no está garantizado.
</details>

**Paso 3 ✋ — Mide la diversidad.** La celda cuenta, para cada par de modelos, en cuántos pacientes se equivocan **los dos** a la vez, y cuántos errores tiene cada uno.

In [ ]:
err = pred.values != y_cm[:, None]
print("Errores de cada modelo:", dict(zip(pred.columns, err.sum(axis=0))))
for i in range(3):
    for j in range(i + 1, 3):
        print(f"{pred.columns[i]:>9} y {pred.columns[j]:<6} fallan juntos en {np.sum(err[:, i] & err[:, j]):>2} pacientes")
print("Los tres fallan juntos en", np.sum(err.all(axis=1)), "pacientes")

✍️ *¿Qué ves?*

<details><summary><b>👀 Ver respuesta</b></summary>

La logística falla en 11 pacientes, KNN en 22 y el árbol en 47. Los errores **sí** son bastante distintos (logística y árbol solo comparten 6, y los tres juntos solo 4). El problema no es la falta de diversidad sino que el árbol es **mucho peor**: cuando el árbol y KNN se equivocan juntos (17 pacientes), le ganan la votación a la logística, que en muchos de esos casos tenía razón.
</details>

**Paso 4 ✏️ — Cambia al miembro débil.** Reemplaza el árbol por un **Naive Bayes** gaussiano (Lección 5.6), que tiene una exactitud parecida pero "piensa" de forma muy distinta. ¿Mejora el comité?

In [ ]:
pred["NaiveBayes"] = cross_val_predict(GaussianNB(), X_cm, y_cm, cv=5)
print(f"Exactitud Naive Bayes solo       : {np.mean(pred['NaiveBayes'] == y_cm):.3f}")

COMITE = ["Logística", "KNN", "NaiveBayes"]     # ✏️ prueba otras combinaciones de 3 (o de 5)
print(f"Voto {' + '.join(COMITE)}: {exactitud_voto(pred, COMITE):.3f}")

<details><summary><b>👀 Ver respuesta</b></summary>

Naive Bayes solo tiene ≈0.938, y el voto Logística + KNN + Naive Bayes da ≈**0.965**: igual que con el árbol. Tampoco supera a la logística. ✏️ Si pruebas el comité de 4 (`["Logística", "KNN", "Árbol", "NaiveBayes"]`) sale ≈**0.961**: con un número par hay empates, y la celda resuelve los 2 contra 2 a favor de la clase 0 ("maligno").

Lo que este paso enseña: con un miembro mucho mejor que el resto, el voto **simple** desperdicia esa ventaja. Para eso existen el voto **ponderado** y el **stacking**, donde un modelo aprende a quién creerle más (Lección 6.5).
</details>

**Paso 5 ✋ — Conclusión para el hospital.** Escribe en tres frases qué le recomendarías al hospital: ¿usar la logística sola o un comité? ¿Por qué?

✍️ *Mi recomendación:*

<details><summary><b>👀 Ver respuesta</b></summary>

Una respuesta razonable: *"Con estos datos, la regresión logística sola es la mejor opción: tiene la mayor exactitud (≈98 %) y es la más fácil de explicar. Un comité de votación simple no la mejora porque los otros modelos son claramente peores y, cuando se equivocan juntos, le quitan la razón. Vale la pena explorar ensambles más inteligentes (Random Forest, boosting o stacking, Lecciones 6.2–6.5), pero solo si mejoran de forma clara en validación cruzada."*

Un ensamble no es magia: **hay que medirlo**, igual que cualquier modelo.
</details>

---
## 📚 Para profundizar
- 📘 **Libro:** James Surowiecki, *The Wisdom of Crowds* (*Cien mejor que uno*): de dónde viene la historia del buey y cuándo la multitud falla.
- 📺 **Video:** StatQuest, "Bagging" y "Random Forests Part 1" (inglés, con subtítulos).
- 📘 **Documentación:** scikit-learn, guía de usuario, sección *Ensembles: Gradient boosting, random forests, bagging, voting, stacking*.

---
⬅️ **Anterior:** [5.8 Proyecto del módulo: Comité de crédito 🏦](../../nivel-2-aprendiz/m05-modelos-clasicos/5.8-proyecto-del-modulo.md) · ➡️ **Siguiente:** [6.2 Bagging y Random Forest: un bosque de árboles distintos](6.2-bagging-y-random-forest.ipynb)